In [1]:
#import
import numpy as np
import pandas as pd
from rdkit import Chem
import seaborn as sns
import matplotlib.pyplot as plt
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem, DataStructs

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_validate,cross_val_score, KFold
import optuna

In [2]:
def generate_2d_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return {desc_name: func(mol) for desc_name, func in Descriptors.descList}

In [3]:
def generate_3d_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None 
    mol = Chem.AddHs(mol)  
    try:
        AllChem.EmbedMolecule(mol, AllChem.ETKDG())
        AllChem.UFFOptimizeMolecule(mol)
        descriptors_3d = {
            "radius_of_gyration": rdMolDescriptors.CalcRadiusOfGyration(mol),
            "asphericity": rdMolDescriptors.CalcAsphericity(mol),
            "spherocity_index": rdMolDescriptors.CalcSpherocityIndex(mol),
            "eccentricity": rdMolDescriptors.CalcEccentricity(mol),
            "NPR1": rdMolDescriptors.CalcNPR1(mol),
            "NPR2": rdMolDescriptors.CalcNPR2(mol),
            "PMI1": rdMolDescriptors.CalcPMI1(mol),
            "PMI2": rdMolDescriptors.CalcPMI2(mol),
            "PMI3": rdMolDescriptors.CalcPMI3(mol)}
        return descriptors_3d
    except:
        return None 

In [4]:
datafile = pd.read_csv("hf://datasets/openadmet/cyp-challenge-train-test/cyp-challenge-TRAIN_inhibition.csv")

In [5]:
smiles_list = datafile["SMILES"].to_list()

In [6]:
data = []
for smiles in smiles_list:
    descriptors_2d = generate_2d_descriptors(smiles)
    descriptors_3d = generate_3d_descriptors(smiles)
    if descriptors_2d and descriptors_3d:
        combined_descriptors = {**descriptors_2d, **descriptors_3d}
        combined_descriptors["SMILES"] = smiles
        data.append(combined_descriptors)

In [7]:
df = pd.DataFrame(data)

In [8]:
data = datafile.merge(df,on="SMILES").drop_duplicates()

In [21]:
CYP_ISOFORMS = ["CYP1A2", "CYP2C9", "CYP2D6", "CYP3A4"]
value_cols = [f"{cyp}_pIC50_direct_inhibition" for cyp in CYP_ISOFORMS]
std_cols = [f"{cyp}_pIC50_direct_inhibition_std" for cyp in CYP_ISOFORMS]
conf_high_cols = [f"{cyp}_pIC50_direct_inhibition_conf_high" for cyp in CYP_ISOFORMS]
conf_low_cols = [f"{cyp}_pIC50_direct_inhibition_conf_low" for cyp in CYP_ISOFORMS]

In [23]:
#data.drop(value_cols,axis=1,inplace=True)
#data.drop(std_cols,axis=1,inplace=True)
data.drop(conf_high_cols,axis=1,inplace=True)
data.drop(conf_low_cols,axis=1,inplace=True)

In [25]:
data.to_csv("Feature_train.csv",index=False)